# Data cleaning for the Index of Multiple Deprivation dataset

## Data sources

The Index of Multiple Deprivation (IMD) dataset was published by the Ministry of Housing, Communities & Local Government (MHCLG) and was constructed by Oxford Consultants for Social Inclusion (OCSI) and Deprivation.org. This dataset was downloaded from the GOV.UK website, as an Excel file (.xlsx), using the link 'https://www.gov.uk/government/statistics/english-indices-of-deprivation-2025' on 9th June 2026. The official statistics on the source website were published on 30 October 2025 and last updated on 17 November 2025.


An additional dataset has also been utilised, to support the preprocessing of the IMD dataset, the necessity of which is detailed fully later in this notebook. In brief, the dataset is required for standardising the location data of the IMD dataset, to allow comparison to other datasets (such as the ‘Police recorded crime open data tables, year ending March 2013 onwards’ dataset) and improve overall ease of understanding.

The 'Local Authority District (December 2024) to LAU1 to ITL3 to ITL2 to ITL1 (January 2025) Lookup in the UK' dataset was published by the Office for National Statistics (ONS). The dataset was downloaded from the National Data Library website, as a csv file (.csv), using the link 'https://www.data.gov.uk/dataset/2b47adcc-62b6-4dd3-a6cb-271b3035e9fd/local-authority-district-december-2024-to-lau1-to-itl3-to-itl2-to-itl1-january-2025-lookup-in-t' on 9th June 2026. The dataset was last updated on 09 December 2025 and the csv file was added to the website on 12 January 2025. 

## Requirements:
 - pandas
 - numpy
 - openpyxl

In [11]:
import pandas as pd
import numpy as np
import openpyxl as op

## Pre-cleaning data check

In [12]:
# Uploading the IMD dataset
df = pd.read_excel('2025_index_of_multiple_deprivation.xlsx',
                  sheet_name='IMD25')

# Visually checking the first and last 5 rows of data in the IMD dataset
# Including key information, such as total number of rows, column names and a sample of data
print(f'First five rows of data:\n{df.head(5)}')
print(f'\nLast five rows of data:\n{df.tail(5)}')

# Shape of dataset - to confirm the number of rows and columns
print(f'\nPre-cleaning IMD dataset shape: {df.shape}')

# Summary statistics pre-cleaning
# Notice that this only provides stats for numerical data (the IMD rank and IMD decile)
print(f'\nPre-cleaning IMD summary statistics:\n{df.describe()}')


First five rows of data:
  LSOA code (2021)           LSOA name (2021)  \
0        E01000001        City of London 001A   
1        E01000002        City of London 001B   
2        E01000003        City of London 001C   
3        E01000005        City of London 001E   
4        E01000006  Barking and Dagenham 016A   

  Local Authority District code (2024) Local Authority District name (2024)  \
0                            E09000001                       City of London   
1                            E09000001                       City of London   
2                            E09000001                       City of London   
3                            E09000001                       City of London   
4                            E09000002                 Barking and Dagenham   

   Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)  \
0                                              26525                     
1                                              31203     

## Data Cleaning

Data quality checks, including:
 - Missing values
 - Data types
 - Unrealistic/unexpected values
 - Duplicate values

In [13]:
# Copying the IMD dataset, in order to preserve the original dataset
df_copy = df.copy()

# Checking for missing values, with none found across all columns
missing_values = df_copy.isnull().sum()
print(f'Total missing values for the Index of Multiple Deprivation \
(IMD) dataset: {missing_values.sum()}')

# Investigating data types and assessing if they are the most appropriate for the values
print(f'\nAll data types for the IMD dataset: \nLSOA Code: {df_copy['LSOA code (2021)'].dtypes}\
\nLSOA Name: {df_copy['LSOA name (2021)'].dtypes}\nLAD Code: {df_copy['Local Authority District code (2024)'].dtypes}\
\nLAD Name: {df_copy['Local Authority District name (2024)'].dtypes}\nIMD Rank: \
{df_copy['Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)'].dtypes}\
\nIMD Decile: {df_copy['Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOA'].dtypes}')

# Checking that the IMD rank is between 1 and 33755, and that the IMD decile is between 1 and 10
if df_copy[(df_copy['Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)'] < 1) | \
    (df_copy['Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)']> 33755)].shape[0]:
    print(f'\nUnexpected IMD Rank values:\n{df_copy[(df_copy['Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)']\
    < 1) | (df_copy['Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)']> 33755)].values}')
else:
    print('\nNo unexpected IMD Rank values.')

if df_copy[(df_copy['Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOA'] < 1) | \
    (df_copy['Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOA']> 10)].shape[0]:
    print(f'\nUnexpected IMD Decile values:\n{df_copy[(df_copy['Index of Multiple Deprivation (IMD) \
    Decile (where 1 is most deprived 10% of LSOA'] < 1) | (df_copy['Index of Multiple Deprivation (IMD) Decile \
    (where 1 is most deprived 10% of LSOA']> 10)].values}')
else:
    print('\nNo unexpected IMD Decile values.')

Total missing values for the Index of Multiple Deprivation (IMD) dataset: 0

All data types for the IMD dataset: 
LSOA Code: str
LSOA Name: str
LAD Code: str
LAD Name: str
IMD Rank: int64
IMD Decile: int64

No unexpected IMD Rank values.

No unexpected IMD Decile values.


All data types are suitable:
 - The LSOA code, LSOA name, LAD code and LAD name columns are all strings, which is appropriate as they contain either solely letters or both numbers and letters
 - The IMD rank and IMD decile are integers, which is the most appropriate data type for these numerical values 

The LSOA code, LSOA name and LAD name columns have been removed from the IMD dataset, in the code cell below. This decision was made because these columns are superfluous to our needs, as the LAD code will be used to map to the ITLs, and the LSOA codes, LSOA names and LAD names are, therefore, not required. 

In [36]:
# Remove unnecessary columns, such as the LSOA code, LSOA name and LAD name
df_imd = df.drop(['LSOA code (2021)', 'LSOA name (2021)', \
                  'Local Authority District name (2024)'], axis=1)

# Shape of dataset - to confirm the columns have been deleted successfully
print(f'IMD dataset shape: {df_imd.shape}')


# Check for outliers in the dataset
# Check for outliers with IQR method and remove outliers?



# Check for duplicates
# This can be done using the IMD rank, as each value should be unique
# With unique values ranging from 1 to 33755
# A value of 33755 would indicate that all values are unique
unique_values = np.count_nonzero(df_imd['Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)']\
.unique())
print(f'\nNumber of unique IMD rank values: {unique_values}')

IMD dataset shape: (33755, 3)

Number of unique values: 33755


## Standardising location data

Mapping Local Authority District (LAD) codes against International Territorial Levels (ITLs), specifically ITL1 and ITL2.

I decided to use the LAD codes to map to the ITLs, rather than the LSOAs, as this approach is facilitated by the Office for National Statistics (ONS), which have published a 'Local Authority District (December 2024) to LAU1 to ITL3 to ITL2 to ITL1 (January 2025) Lookup in the UK' dataset. The LAD codes and names in the IMD dataset are for 2024, whilst the ITLs we decided to use are for 2025. Therefore, the ONS dataset chosen is the most appropriate for this task.

The ITL1 and ITL2 regions can then be utilised across the datasets for this project, ensuring that location data is directly comparable. This is necessary as, for example, the police dataset utilises police districts that are not directly comparable to LSOAs or LADs.

In [15]:
# Uploading the LAD to ITL lookup dataset
df_itl = pd.read_csv('LAD_2024_to_ITL_2025.csv')

# Checking the LAD to ITL lookup dataset shape
print(f'\nLAD to ITL dataset shape: {df_itl.shape}\n')

# Visually checking the head of the dataset - for instance, column names and data are as expected
print(df_itl.head(3))



LAD to ITL dataset shape: (362, 11)

  ITL125CD              ITL125NM ITL225CD     ITL225NM ITL325CD  \
0      TLC  North East (England)     TLC3  Tees Valley    TLC31   
1      TLC  North East (England)     TLC3  Tees Valley    TLC31   
2      TLC  North East (England)     TLC3  Tees Valley    TLC32   

                          ITL325NM   LAU125CD          LAU125NM    LAD24CD  \
0  Hartlepool and Stockton-on-Tees  E06000001        Hartlepool  E06000001   
1  Hartlepool and Stockton-on-Tees  E06000004  Stockton-on-Tees  E06000004   
2                   South Teesside  E06000002     Middlesbrough  E06000002   

            LAD24NM  ObjectId  
0        Hartlepool         1  
1  Stockton-on-Tees         2  
2     Middlesbrough         3  


The LAD_2024_to_ITL_2025 ONS dataset contains location data for England, Wales, Scotland and Northern Ireland, whereas the Index for Multiple Deprivation dataset only contains data for England. Therefore, I have decided to use a left merge on the LAD codes of the IMD dataset and a right merge on the LAD codes of the LAD to ITL dataset, to retain all location data from the IMD dataset and not pull across any ITLs that aren't required (for example, Wales, Scotland and Northern Ireland) from the LAD to ITL dataset.

In [37]:
# Merging the IMD dataset and the LAD to ITL lookup dataset
df_clean = df_imd.merge(df_itl, left_on='Local Authority District code (2024)', right_on='LAD24CD')

# Visually checking the head and tail of the merged dataset
print(df_clean.head(3))
print(df_clean.tail(3))

# Checking the shape of the merged dataset - expected shape is (33755, 14)
print(f'\nMerged dataset shape: {df_clean.shape}')

# Check for duplicates - to ensure that no duplication occured in the merge
# Using the IMD rank, as each value should be unique
# With unique values ranging from 1 to 33755
# A value of 33755 would indicate that all values are unique
unique_values = np.count_nonzero(df_clean['Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)']\
.unique())
print(f'\nNumber of unique IMD rank values: {unique_values}')

# Checking that the two columns for LAD codes ('Local Authority District code \
# (2024)' and 'LAD24CD' are identical across all rows)
# This acts as another check that the merge was successful
# A value of 33755 would suggest that the data of the columns are completely identical
print(f'\nNumber of rows that have identical LAD codes: \
{(df_clean['Local Authority District code (2024)'] == df_clean['LAD24CD']).count()}')


  Local Authority District code (2024)  \
0                            E09000001   
1                            E09000001   
2                            E09000001   

   Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)  \
0                                              26525                     
1                                              31203                     
2                                              25913                     

   Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOA  \
0                                                  8                                  
1                                                 10                                  
2                                                  8                                  

  ITL125CD ITL125NM ITL225CD             ITL225NM ITL325CD  \
0      TLI   London     TLI3  Inner London - West    TLI35   
1      TLI   London     TLI3  Inner London - West    TLI35   

The decision was made to delete the columns listed below, from the LAD to ITL lookup dataset, and retain only the IMD rank, IMD decile, ITL1 name and ITL2 name columns remaining. The ITL3 values, Local Authority Unit (LAU) values, ObjectId and LAD names are not required for this project. The ITL1 and ITL2 codes are redundent because the ITL1 and ITL2 names are sufficient. Additionally, both of the LAD code columns were deleted, as these are now unnecessary after merging the datasets and mapping the LAD codes to ITL1 and ITL2 values.

 - 'ITL125CD' (ITL1 code)
 - 'ITL225CD' (ITL2 code)
 - 'ITL325CD' (ITL3 code)
 - 'ITL325NM' (ITL3 name)
 - 'LAU125CD' (LAU code)
 - 'LAU125NM' (LAU name)
 - 'LAD24NM'  (LAD name)
 - 'ObjectId' (ObjectId)
 - 'Local Authority District code (2024)' (LAD code)
 - 'LAD24CD' (LAD code)


In [39]:
# Delete unnecessary columns (v stands for version)
df_clean_v2 = df_clean.drop(['ITL125CD', 'ITL225CD', 'ITL325CD', \
                             'ITL325NM', 'LAU125CD', 'LAU125NM', 'ObjectId', 'LAD24NM', \
                             'Local Authority District code (2024)', 'LAD24CD'], axis=1)

# Checking the shape of the dataset - to check that columns were successfully deleted
# Expected shape is (33755, 6)
print(f'\nUpdated dataset shape: {df_clean_v2.shape}\n')

# Visually checking the head of the dataset - to ensure data is as expected
df_clean_v2.head()


Updated dataset shape: (33755, 4)



,Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived),Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOA,ITL125NM,ITL225NM
0,26525,8,London,Inner London - West
1,31203,10,London,Inner London - West
2,25913,8,London,Inner London - West
3,14807,5,London,Inner London - West
4,10917,4,London,Outer London - East and North East


## Renaming columns for ease of use

In [ ]:
# rename columns .str.replace('','')
df_clean_final = 


## Clean dataset

In [ ]:
# Shape of dataset - post-cleaning - to confirm the number of rows and columns
print(f'\nPost-cleaning IMD dataset shape:\n{df_clean_final.shape}')

# Summary statistics post-cleaning
print(f'\nPost-cleaning summary statistics:\n{df_clean_final.describe()}')

# Save the clean dataset as an Excel file
df_clean_final.to_csv('clean_index_of_multiple_deprivation', index=False)